In [4]:
import requests
import pandas as pd
import json
import re
import time
from datetime import datetime

In [5]:
BASE_CONAF = "https://esri.ciren.cl/server/rest/services/INCENDIOS_FORESTALES/FeatureServer"

meta = requests.get(
    BASE_CONAF,
    params={"f": "json"}
).json()

capas_conaf = pd.DataFrame(meta["layers"])

capas_conaf

,id,name,parentLayerId,defaultVisibility,subLayerIds,minScale,maxScale,type,geometryType
0,0,INCENDIOS FORESTALES 2010 - 2011,-1,False,None,0,0,Feature Layer,esriGeometryPoint
1,1,INCENDIOS FORESTALES 2011 - 2012,-1,False,None,0,0,Feature Layer,esriGeometryPoint
2,2,INCENDIOS FORESTALES 2012 - 2013,-1,False,None,0,0,Feature Layer,esriGeometryPoint
3,3,INCENDIOS FORESTALES 2013 - 2014,-1,False,None,0,0,Feature Layer,esriGeometryPoint
4,4,INCENDIOS FORESTALES 2014 - 2015,-1,False,None,0,0,Feature Layer,esriGeometryPoint
5,5,INCENDIOS FORESTALES 2015 - 2016,-1,False,None,0,0,Feature Layer,esriGeometryPoint
6,6,INCENDIOS FORESTALES 2016 - 2017,-1,False,None,0,0,Feature Layer,esriGeometryPoint
7,7,INCENDIOS FORESTALES 2017 - 2018,-1,False,None,0,0,Feature Layer,esriGeometryPoint
8,8,INCENDIOS FORESTALES 2018 - 2019,-1,False,None,0,0,Feature Layer,esriGeometryPoint
9,9,INCENDIOS FORESTALES 2019 - 2020,-1,False,None,0,0,Feature Layer,esriGeometryPoint


In [6]:
LAYER = 16

meta_capa = requests.get(
    f"{BASE_CONAF}/{LAYER}",
    params={"f": "json"}
).json()

campos = pd.DataFrame(meta_capa["fields"])

campos[["name", "alias", "type"]]

,name,alias,type
0,objectid,OBJECTID,esriFieldTypeOID
1,temporada,TEMPORADA,esriFieldTypeString
2,region,REGION,esriFieldTypeString
3,provincia,PROVINCIA,esriFieldTypeString
4,comuna,COMUNA,esriFieldTypeString
5,codreg,CODREG,esriFieldTypeDouble
6,n_mbito,N_MBITO,esriFieldTypeString
7,n_mero_re,N_MERO_RE,esriFieldTypeDouble
8,nombre,NOMBRE,esriFieldTypeString
9,utm_e,UTM_E,esriFieldTypeDouble


In [7]:
url = f"{BASE_CONAF}/{LAYER}/query"

params = {
    "where": "UPPER(region) LIKE '%VALPARA%'",
    "outFields": "*",
    "returnGeometry": "false",
    "f": "json"
}

respuesta = requests.get(url, params=params).json()

df_conaf = pd.DataFrame(
    [x["attributes"] for x in respuesta["features"]]
)

print(df_conaf.shape)
display(df_conaf.head())
print(df_conaf.columns.tolist())

(536, 32)


,objectid,temporada,region,provincia,comuna,codreg,n_mbito,n_mero_re,nombre,utm_e,...,arbolado,matorral,pastizal,subtotal_v,agr_cola,desechos,subtotal_o,superficie,fh_inicio,fh_extinci
0,3484,2024-2025,Valparaíso,Marga Marga,Villa Alemana,5.0,Conaf,2.0,FUNDO EL RINCON,282083.0,...,0.00,0.0,0.00,0.00,0.0,0.05,0.05,0.05,29-jul-2024 12:05,29-jul-2024 14:40
1,3485,2024-2025,Valparaíso,Los Andes,Los Andes,5.0,Conaf,3.0,PLANTA LOS QUILOS,369112.0,...,0.00,0.0,0.03,0.03,0.0,0.00,0.00,0.03,29-jul-2024 13:15,31-jul-2024 17:13
2,3486,2024-2025,Valparaíso,San Antonio,Santo Domingo,5.0,Conaf,4.0,EL CONVENTO - LAS VIñAS,256947.0,...,0.00,0.0,0.00,0.00,0.0,0.03,0.03,0.03,1-ago-2024 10:15,1-ago-2024 11:20
3,3487,2024-2025,Valparaíso,Marga Marga,Olmué,5.0,Conaf,5.0,QUEBRADA EL MAQUI,303385.0,...,0.24,0.0,0.00,0.24,0.0,0.00,0.00,0.24,24-ago-2024 13:20,25-ago-2024 18:00
4,3488,2024-2025,Valparaíso,Marga Marga,Limache,5.0,Conaf,6.0,CALLEJON OLMOS,288982.0,...,0.00,0.0,0.00,0.00,0.0,0.04,0.04,0.04,4-sep-2024 09:19,4-sep-2024 12:30


['objectid', 'temporada', 'region', 'provincia', 'comuna', 'codreg', 'n_mbito', 'n_mero_re', 'nombre', 'utm_e', 'utm_n', 'huso_op_', 'inicio_cer', 'combustibl', 'causa_gene', 'causa_espe', 'pino_0_a_1', 'pino_11_a', 'pino_18_o', 'eucalipto', 'otras_plan', 'subtotal_p', 'arbolado', 'matorral', 'pastizal', 'subtotal_v', 'agr_cola', 'desechos', 'subtotal_o', 'superficie', 'fh_inicio', 'fh_extinci']


In [8]:
df_conaf["fh_inicio_dt"] = pd.to_datetime(
    df_conaf["fh_inicio"],
    errors="coerce",
    dayfirst=True
)

print("Primera fecha:", df_conaf["fh_inicio_dt"].min())
print("Última fecha:", df_conaf["fh_inicio_dt"].max())
print("Incendios:", len(df_conaf))

display(
    df_conaf[
        ["temporada", "comuna", "fh_inicio", "superficie"]
    ].head(20)
)

Primera fecha: 2024-07-29 12:05:00
Última fecha: 2025-05-30 14:38:00
Incendios: 536


C:\Users\Tomy\AppData\Local\Temp\ipykernel_21696\2188185220.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_conaf["fh_inicio_dt"] = pd.to_datetime(


,temporada,comuna,fh_inicio,superficie
0,2024-2025,Villa Alemana,29-jul-2024 12:05,0.05
1,2024-2025,Los Andes,29-jul-2024 13:15,0.03
2,2024-2025,Santo Domingo,1-ago-2024 10:15,0.03
3,2024-2025,Olmué,24-ago-2024 13:20,0.24
4,2024-2025,Limache,4-sep-2024 09:19,0.04
5,2024-2025,Panquehue,8-sep-2024 21:34,0.05
6,2024-2025,Concón,14-sep-2024 18:11,0.05
7,2024-2025,Quilpué,1-oct-2024 15:02,0.01
8,2024-2025,Quilpué,14-oct-2024 15:43,0.04
9,2024-2025,Santo Domingo,15-oct-2024 13:30,0.32


In [9]:
resumen_conaf = []
datos_conaf = []

for _, capa in capas_conaf.iterrows():
    layer_id = capa["id"]
    nombre = capa["name"]

    url = f"{BASE_CONAF}/{layer_id}/query"

    params = {
        "where": "1=1",
        "outFields": "*",
        "returnGeometry": "false",
        "f": "json"
    }

    r = requests.get(url, params=params).json()

    if "features" not in r:
        print("Error:", nombre, r)
        continue

    df = pd.DataFrame([x["attributes"] for x in r["features"]])

    if "region" in df.columns:
        df = df[
            df["region"]
            .astype(str)
            .str.contains("Valpara", case=False, na=False)
        ]

    if len(df) == 0:
        continue

    fecha_col = "fh_inicio" if "fh_inicio" in df.columns else None

    if fecha_col:
        fechas = pd.to_datetime(
            df[fecha_col],
            errors="coerce",
            dayfirst=True
        )

        inicio = fechas.min()
        fin = fechas.max()
    else:
        inicio = None
        fin = None

    resumen_conaf.append({
        "layer": layer_id,
        "temporada": nombre,
        "registros_valparaiso": len(df),
        "fecha_min": inicio,
        "fecha_max": fin
    })

    df["layer"] = layer_id
    datos_conaf.append(df)

resumen_conaf = pd.DataFrame(resumen_conaf)

display(resumen_conaf)

C:\Users\Tomy\AppData\Local\Temp\ipykernel_21696\2888091337.py:38: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  fechas = pd.to_datetime(
C:\Users\Tomy\AppData\Local\Temp\ipykernel_21696\2888091337.py:38: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  fechas = pd.to_datetime(
C:\Users\Tomy\AppData\Local\Temp\ipykernel_21696\2888091337.py:38: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  fechas = pd.to_datetime(


,layer,temporada,registros_valparaiso,fecha_min,fecha_max
0,5,INCENDIOS FORESTALES 2015 - 2016,743,NaT,NaT
1,7,INCENDIOS FORESTALES 2017 - 2018,953,NaT,NaT
2,8,INCENDIOS FORESTALES 2018 - 2019,887,NaT,NaT
3,9,INCENDIOS FORESTALES 2019 - 2020,608,NaT,NaT
4,10,INCENDIOS FORESTALES 2020 - 2021,347,NaT,NaT
5,11,INCENDIOS FORESTALES 2021 - 2022,381,2021-07-05 14:10:00.000000,2022-06-21 18:45:00.000000
6,12,INCENDIOS FORESTALES 2022 - 2023,362,2022-07-07 15:35:00.000000,2023-06-16 15:12:00.000000
7,14,INCENDIOS FORESTALES 2023 - 2024,498,1970-01-01 00:28:15.772800,1970-01-01 00:28:39.532800
8,16,INCENDIOS FORESTALES 2024 - 2025,536,2024-07-29 12:05:00.000000,2025-05-30 14:38:00.000000


In [10]:
resumen_esquemas = []

for _, capa in capas_conaf.iterrows():
    layer_id = capa["id"]
    nombre = capa["name"]

    meta = requests.get(
        f"{BASE_CONAF}/{layer_id}",
        params={"f": "json"}
    ).json()

    campos = [x["name"] for x in meta.get("fields", [])]

    relacionados = [
        c for c in campos
        if any(p in c.lower() for p in [
            "fecha", "inicio", "region", "comuna", "temporada"
        ])
    ]

    resumen_esquemas.append({
        "layer": layer_id,
        "temporada": nombre,
        "campos_relevantes": relacionados
    })

pd.set_option("display.max_colwidth", None)
display(pd.DataFrame(resumen_esquemas))

,layer,temporada,campos_relevantes
0,0,INCENDIOS FORESTALES 2010 - 2011,"[temporada, comuna, region, inicio_c, inicio_in]"
1,1,INCENDIOS FORESTALES 2011 - 2012,"[temporada, comuna, region, inicio_c, inicio_in]"
2,2,INCENDIOS FORESTALES 2012 - 2013,"[temporada, comuna, region, inicio_c, inicio_in]"
3,3,INCENDIOS FORESTALES 2013 - 2014,"[temporada, region, comuna, inicio_c, inicio_in]"
4,4,INCENDIOS FORESTALES 2014 - 2015,"[temporada, region, comuna, inicio_c, inicio_in]"
5,5,INCENDIOS FORESTALES 2015 - 2016,"[temporada, region, comuna, inicio_c, inicio_in]"
6,6,INCENDIOS FORESTALES 2016 - 2017,"[temporada, region, comuna, inicio_c, inicio_in]"
7,7,INCENDIOS FORESTALES 2017 - 2018,"[temporada, inicio_c, region, comuna, inicio_in]"
8,8,INCENDIOS FORESTALES 2018 - 2019,"[region, comuna, inicio_c, inicio_in, temporada]"
9,9,INCENDIOS FORESTALES 2019 - 2020,"[region, comuna, inicio_c, inicio_in, temporada]"


In [11]:
auditoria_fechas = []

for _, capa in capas_conaf.iterrows():
    layer_id = capa["id"]
    temporada = capa["name"]

    url = f"{BASE_CONAF}/{layer_id}/query"

    params = {
        "where": "1=1",
        "outFields": "*",
        "returnGeometry": "false",
        "resultRecordCount": 5,
        "f": "json"
    }

    r = requests.get(url, params=params).json()

    if "features" not in r or len(r["features"]) == 0:
        continue

    df = pd.DataFrame([x["attributes"] for x in r["features"]])

    columnas_fecha = [
        c for c in ["inicio_c", "inicio_in", "inicio_cer", "fh_inicio", "fh_extinci"]
        if c in df.columns
    ]

    for c in columnas_fecha:
        valores = df[c].dropna().head(3).tolist()

        auditoria_fechas.append({
            "layer": layer_id,
            "temporada": temporada,
            "campo": c,
            "ejemplos": valores
        })

pd.set_option("display.max_colwidth", None)
display(pd.DataFrame(auditoria_fechas))

,layer,temporada,campo,ejemplos
0,0,INCENDIOS FORESTALES 2010 - 2011,inicio_c,"[SENDERO, CAMINO SECUNDARIO, SENDERO]"
1,0,INCENDIOS FORESTALES 2010 - 2011,inicio_in,"[ , , ]"
2,1,INCENDIOS FORESTALES 2011 - 2012,inicio_c,"[CAMINO SECUNDARIO, CAMINO SECUNDARIO, CAMINO SECUNDARIO]"
3,1,INCENDIOS FORESTALES 2011 - 2012,inicio_in,"[ , , ]"
4,2,INCENDIOS FORESTALES 2012 - 2013,inicio_c,"[CAMINO PRINCIPAL, CAMINO PRINCIPAL, ]"
5,2,INCENDIOS FORESTALES 2012 - 2013,inicio_in,"[ , , ]"
6,3,INCENDIOS FORESTALES 2013 - 2014,inicio_c,"[Sendero, , ]"
7,3,INCENDIOS FORESTALES 2013 - 2014,inicio_in,"[ , , ]"
8,4,INCENDIOS FORESTALES 2014 - 2015,inicio_c,"[Otros, , ]"
9,4,INCENDIOS FORESTALES 2014 - 2015,inicio_in,"[ , , ]"


In [12]:
auditoria_fechas_2 = []

for _, capa in capas_conaf.iterrows():
    layer_id = capa["id"]
    temporada = capa["name"]

    url = f"{BASE_CONAF}/{layer_id}/query"

    params = {
        "where": "1=1",
        "outFields": "*",
        "returnGeometry": "false",
        "f": "json"
    }

    r = requests.get(url, params=params).json()

    if "features" not in r:
        continue

    df = pd.DataFrame([x["attributes"] for x in r["features"]])

    candidatos = [
        c for c in df.columns
        if any(x in c.lower() for x in [
            "fecha", "inicio", "fh_", "hora"
        ])
    ]

    for c in candidatos:
        s = df[c]

        no_vacios = (
            s.notna()
            & s.astype(str).str.strip().ne("")
        )

        ejemplos = s[no_vacios].head(3).tolist()

        auditoria_fechas_2.append({
            "layer": layer_id,
            "temporada": temporada,
            "campo": c,
            "no_vacios": int(no_vacios.sum()),
            "ejemplos": ejemplos
        })

auditoria_fechas_2 = pd.DataFrame(auditoria_fechas_2)

pd.set_option("display.max_colwidth", None)
display(auditoria_fechas_2)

,layer,temporada,campo,no_vacios,ejemplos
0,0,INCENDIOS FORESTALES 2010 - 2011,inicio_c,4872,"[SENDERO, CAMINO SECUNDARIO, SENDERO]"
1,0,INCENDIOS FORESTALES 2010 - 2011,inicio_in,0,[]
2,1,INCENDIOS FORESTALES 2011 - 2012,inicio_c,5439,"[CAMINO SECUNDARIO, CAMINO SECUNDARIO, CAMINO SECUNDARIO]"
3,1,INCENDIOS FORESTALES 2011 - 2012,inicio_in,0,[]
4,2,INCENDIOS FORESTALES 2012 - 2013,inicio_c,3925,"[CAMINO PRINCIPAL, CAMINO PRINCIPAL, CAMINO PRINCIPAL]"
5,2,INCENDIOS FORESTALES 2012 - 2013,inicio_in,0,[]
6,3,INCENDIOS FORESTALES 2013 - 2014,inicio_c,4365,"[Sendero, Camino principal, Casa o refugio]"
7,3,INCENDIOS FORESTALES 2013 - 2014,inicio_in,0,[]
8,4,INCENDIOS FORESTALES 2014 - 2015,inicio_c,4868,"[Otros, Camino secundario, Casa o refugio]"
9,4,INCENDIOS FORESTALES 2014 - 2015,inicio_in,0,[]


In [13]:
pd.to_datetime(
    df["fh_inicio"],
    unit="ms",
    errors="coerce"
)

0      NaT
1      NaT
2      NaT
3      NaT
4      NaT
        ..
6257   NaT
6258   NaT
6259   NaT
6260   NaT
6261   NaT
Name: fh_inicio, Length: 6262, dtype: datetime64[ns]

In [14]:
pd.to_datetime(
    df["fh_inicio"],
    errors="coerce",
    dayfirst=True
)

C:\Users\Tomy\AppData\Local\Temp\ipykernel_21696\3408885420.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  pd.to_datetime(


0                      NaT
1      2024-09-16 08:14:00
2                      NaT
3      2025-02-10 18:06:00
4      2025-02-10 18:22:00
               ...        
6257                   NaT
6258                   NaT
6259                   NaT
6260                   NaT
6261                   NaT
Name: fh_inicio, Length: 6262, dtype: datetime64[ns]

In [15]:
def convertir_fecha_conaf(df, layer_id):
    if layer_id in [9, 10]:
        # 2019-2020 y 2020-2021
        return pd.to_datetime(
            df["inicio_in"],
            errors="coerce",
            dayfirst=True
        )

    elif layer_id in [11, 12]:
        # 2021-2022 y 2022-2023
        return pd.to_datetime(
            df["fh_inicio"],
            errors="coerce",
            dayfirst=True
        )

    elif layer_id == 14:
        # 2023-2024: Unix timestamp en milisegundos
        return pd.to_datetime(
            pd.to_numeric(df["fh_inicio"], errors="coerce"),
            unit="ms",
            errors="coerce"
        )

    elif layer_id == 16:
        # 2024-2025: texto
        return pd.to_datetime(
            df["fh_inicio"],
            errors="coerce",
            dayfirst=True
        )

    return pd.Series(pd.NaT, index=df.index)

In [16]:
layers_utiles = [9, 10, 11, 12, 14, 16]

lista_conaf = []

for layer_id in layers_utiles:

    url = f"{BASE_CONAF}/{layer_id}/query"

    params = {
        "where": "1=1",
        "outFields": "*",
        "returnGeometry": "false",
        "f": "json"
    }

    r = requests.get(url, params=params).json()

    df = pd.DataFrame([
        x["attributes"]
        for x in r["features"]
    ])

    # Normalizar fecha
    df["fecha_inicio"] = convertir_fecha_conaf(df, layer_id)

    # Región de Valparaíso
    if "region" in df.columns:
        df = df[
            df["region"]
            .astype(str)
            .str.contains("Valpara", case=False, na=False)
        ]

    df["layer"] = layer_id

    lista_conaf.append(df)

df_conaf_historico = pd.concat(
    lista_conaf,
    ignore_index=True
)

print(df_conaf_historico.shape)

display(
    df_conaf_historico[
        ["temporada", "region", "comuna", "fecha_inicio"]
    ].head(20)
)

C:\Users\Tomy\AppData\Local\Temp\ipykernel_21696\1335649984.py:4: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  return pd.to_datetime(
C:\Users\Tomy\AppData\Local\Temp\ipykernel_21696\1335649984.py:4: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  return pd.to_datetime(
C:\Users\Tomy\AppData\Local\Temp\ipykernel_21696\1335649984.py:12: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  return pd.to_datetime(
C:\Users\Tomy\AppData\Local\Temp\ipykernel_21696\1335649984.py:12: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil

(2732, 54)


C:\Users\Tomy\AppData\Local\Temp\ipykernel_21696\1335649984.py:28: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  return pd.to_datetime(


,temporada,region,comuna,fecha_inicio
0,2019-2020,Valparaíso,Hijuelas,2019-07-15 16:30:00
1,2019-2020,Valparaíso,Calera,NaT
2,2019-2020,Valparaíso,Quillota,NaT
3,2019-2020,Valparaíso,Concón,NaT
4,2019-2020,Valparaíso,La Ligua,NaT
5,2019-2020,Valparaíso,La Ligua,NaT
6,2019-2020,Valparaíso,El Quisco,NaT
7,2019-2020,Valparaíso,Panquehue,NaT
8,2019-2020,Valparaíso,La Cruz,NaT
9,2019-2020,Valparaíso,Quillota,NaT


In [20]:
resumen = (
    df_conaf_historico
    .groupby("temporada")
    .agg(
        incendios=("comuna", "size"),
        con_fecha=("fecha_inicio", "count"),
        fecha_min=("fecha_inicio", "min"),
        fecha_max=("fecha_inicio", "max")
    )
    .reset_index()
)

display(resumen)

,temporada,incendios,con_fecha,fecha_min,fecha_max
0,2019-2020,608,351,2019-07-15 16:30:00,2020-06-11 13:43:00
1,2020-2021,347,170,2020-07-31 14:27:00,2021-06-29 12:15:00
2,2021-2022,381,214,2021-07-05 14:10:00,2022-06-21 18:45:00
3,2022-2023,362,180,2022-07-07 15:35:00,2023-06-16 15:12:00
4,2023-2024,498,498,2023-09-27 00:00:00,2024-06-28 00:00:00
5,2024-2025,536,289,2024-07-29 12:05:00,2025-05-30 14:38:00


In [21]:
print(
    "Fecha mínima total:",
    df_conaf_historico["fecha_inicio"].min()
)

print(
    "Fecha máxima total:",
    df_conaf_historico["fecha_inicio"].max()
)

print(
    "Incendios Valparaíso:",
    len(df_conaf_historico)
)

print(
    "Sin fecha:",
    df_conaf_historico["fecha_inicio"].isna().sum()
)

Fecha mínima total: 2019-07-15 16:30:00
Fecha máxima total: 2025-05-30 14:38:00
Incendios Valparaíso: 2732
Sin fecha: 1030


# DMC

DMC_USUARIO = "em.serey@duocuc.cl"
DMC_TOKEN = "Lungo2002?"

In [38]:
from dotenv import load_dotenv
import os

load_dotenv()

DMC_USUARIO = os.getenv("DMC_USUARIO")
DMC_TOKEN = os.getenv("DMC_TOKEN")

print("Usuario cargado:", DMC_USUARIO is not None)
print("Token cargado:", DMC_TOKEN is not None)

Usuario cargado: True
Token cargado: True


In [39]:
ESTACIONES = {
    "320019": "San Felipe Escuela Agrícola",
    "320041": "Viña del Mar Ad. (Torquemada)",
    "330007": "Rodelillo, Ad.",
    "330030": "Santo Domingo, Ad."
}

In [40]:
codigo = "330007"

url = (
    "https://climatologia.meteochile.gob.cl"
    f"/application/servicios/getDatosRecientesEma/{codigo}"
)

r = requests.get(
    url,
    params={
        "usuario": DMC_USUARIO,
        "token": DMC_TOKEN
    }
)

print("HTTP:", r.status_code)

datos = r.json()

print(datos.keys())

print(
    json.dumps(
        datos,
        ensure_ascii=False,
        indent=2
    )[:5000]
)

HTTP: 200
dict_keys(['organismo', 'pais', 'fechaCreacion', 'timezone', 'producto', 'registros', 'status', 'datosEstaciones'])
{
  "organismo": "Dirección Meteorlógica de Chile",
  "pais": "Chile",
  "fechaCreacion": "05-10-2026 21:45",
  "timezone": "UTC",
  "producto": "Datos Recientes de la estación automática últimas 12 horas",
  "registros": 690,
  "status": "Estación con datos disponibles recientemente",
  "datosEstaciones": {
    "estacion": {
      "codigoNacional": "330007",
      "codigoOMM": "85560",
      "codigoOACI": "SCRD",
      "nombreEstacion": "Rodelillo, Ad.",
      "latitud": "-33.06528",
      "longitud": "-71.55639",
      "altura": 335,
      "status": "Estación con datos disponibles recientemente",
      "registros": 690
    },
    "datos": [
      {
        "momento": "2026-10-05 21:15:00",
        "temperatura": "12.1 °C",
        "temperatura02Mts": null,
        "temperatura10Mts": null,
        "temperatura30Mts": null,
        "puntoDeRocio": "10.4 °C",
  

In [48]:
datos_recientes = datos["datosEstaciones"]["datos"]

df_dmc = pd.DataFrame(datos_recientes)

print("Filas:", len(df_dmc))
print("\nColumnas:")
print(df_dmc.columns.tolist())

display(df_dmc.head())

Filas: 2976

Columnas:
['momento', 'temperatura', 'temperatura02Mts', 'temperatura10Mts', 'temperatura30Mts', 'puntoDeRocio', 'temperaturaMinima12Horas', 'temperaturaMaxima12Horas', 'humedadRelativa', 'radiacionGlobalInst', 'presionEstacion', 'presionNivelDelMar', 'presionNivelEstandar', 'aguaCaidaDelMinuto', 'aguaCaida6Horas', 'aguaCaida24Horas', 'direccionDelViento', 'fuerzaDelViento', 'direccionDelVientoPromedio2Minutos', 'fuerzaDelVientoPromedio2Minutos', 'direccionDelVientoPromedio10Minutos', 'fuerzaDelVientoPromedio10Minutos', 'direccionDelViento02MinutosMax', 'fuerzaDelViento02MinutosMax', 'direccionDelViento10MinutosMax', 'fuerzaDelViento10MinutosMax']


,momento,temperatura,temperatura02Mts,temperatura10Mts,temperatura30Mts,puntoDeRocio,temperaturaMinima12Horas,temperaturaMaxima12Horas,humedadRelativa,radiacionGlobalInst,...,direccionDelViento,fuerzaDelViento,direccionDelVientoPromedio2Minutos,fuerzaDelVientoPromedio2Minutos,direccionDelVientoPromedio10Minutos,fuerzaDelVientoPromedio10Minutos,direccionDelViento02MinutosMax,fuerzaDelViento02MinutosMax,direccionDelViento10MinutosMax,fuerzaDelViento10MinutosMax
0,2024-01-01 00:00:00,16.9 °C,None,None,None,11.8 °C,16.7 °C,24.2 °C,71 %,0.000 Watt/m2,...,234 °,0.1 kt,148 °,0.2 kt,167 °,1.5 kt,None,None,None,None
1,2024-01-01 00:15:00,16.4 °C,None,None,None,11.7 °C,16.7 °C,24.2 °C,73 %,0.000 Watt/m2,...,170 °,0.6 kt,158 °,1.3 kt,162 °,2.0 kt,None,None,None,None
2,2024-01-01 00:30:00,15.8 °C,None,None,None,11.6 °C,16.7 °C,24.2 °C,76 %,0.000 Watt/m2,...,128 °,2.0 kt,147 °,2.5 kt,159 °,1.8 kt,None,None,None,None
3,2024-01-01 00:45:00,15.3 °C,None,None,None,11.6 °C,16.7 °C,24.2 °C,78 %,0.000 Watt/m2,...,143 °,1.5 kt,159 °,1.3 kt,159 °,2.3 kt,None,None,None,None
4,2024-01-01 01:00:00,14.8 °C,None,None,None,11.5 °C,16.7 °C,24.2 °C,80 %,0.000 Watt/m2,...,90 °,2.9 kt,144 °,2.5 kt,143 °,1.8 kt,None,None,None,None


In [49]:
columnas_meteo = [
    c for c in df_dmc.columns
    if any(x in c.lower() for x in [
        "momento",
        "temperatura",
        "humedad",
        "viento"
    ])
]

display(df_dmc[columnas_meteo].head(20))

,momento,temperatura,temperatura02Mts,temperatura10Mts,temperatura30Mts,temperaturaMinima12Horas,temperaturaMaxima12Horas,humedadRelativa,direccionDelViento,fuerzaDelViento,direccionDelVientoPromedio2Minutos,fuerzaDelVientoPromedio2Minutos,direccionDelVientoPromedio10Minutos,fuerzaDelVientoPromedio10Minutos,direccionDelViento02MinutosMax,fuerzaDelViento02MinutosMax,direccionDelViento10MinutosMax,fuerzaDelViento10MinutosMax
0,2024-01-01 00:00:00,16.9 °C,None,None,None,16.7 °C,24.2 °C,71 %,234 °,0.1 kt,148 °,0.2 kt,167 °,1.5 kt,None,None,None,None
1,2024-01-01 00:15:00,16.4 °C,None,None,None,16.7 °C,24.2 °C,73 %,170 °,0.6 kt,158 °,1.3 kt,162 °,2.0 kt,None,None,None,None
2,2024-01-01 00:30:00,15.8 °C,None,None,None,16.7 °C,24.2 °C,76 %,128 °,2.0 kt,147 °,2.5 kt,159 °,1.8 kt,None,None,None,None
3,2024-01-01 00:45:00,15.3 °C,None,None,None,16.7 °C,24.2 °C,78 %,143 °,1.5 kt,159 °,1.3 kt,159 °,2.3 kt,None,None,None,None
4,2024-01-01 01:00:00,14.8 °C,None,None,None,16.7 °C,24.2 °C,80 %,90 °,2.9 kt,144 °,2.5 kt,143 °,1.8 kt,None,None,None,None
5,2024-01-01 01:15:00,14.5 °C,None,None,None,16.7 °C,24.2 °C,82 %,135 °,1.6 kt,142 °,2.3 kt,150 °,2.2 kt,None,None,None,None
6,2024-01-01 01:30:00,14.3 °C,None,None,None,16.7 °C,24.2 °C,83 %,117 °,1.8 kt,151 °,2.1 kt,151 °,2.5 kt,None,None,None,None
7,2024-01-01 01:45:00,14.0 °C,None,None,None,16.7 °C,24.2 °C,85 %,179 °,1.7 kt,176 °,2.1 kt,150 °,1.6 kt,None,None,None,None
8,2024-01-01 02:00:00,14.3 °C,None,None,None,16.7 °C,24.2 °C,84 %,182 °,2.2 kt,156 °,1.4 kt,154 °,2.1 kt,None,None,None,None
9,2024-01-01 02:15:00,14.0 °C,None,None,None,16.7 °C,24.2 °C,85 %,138 °,2.4 kt,179 °,1.8 kt,153 °,1.7 kt,None,None,None,None


In [50]:
df_dmc["momento_dt"] = pd.to_datetime(
    df_dmc["momento"],
    errors="coerce"
)

df_dmc = df_dmc.sort_values("momento_dt")

diferencias = (
    df_dmc["momento_dt"]
    .diff()
    .dt.total_seconds()
    .div(60)
)

print("Intervalos más comunes (minutos):")
print(diferencias.value_counts().head(10))

Intervalos más comunes (minutos):
momento_dt
15.0    2975
Name: count, dtype: int64


In [51]:
codigo = "330007"
anio = 2024
mes = 1

url = (
    "https://climatologia.meteochile.gob.cl"
    f"/application/servicios/getDatosRecientesEma/"
    f"{codigo}/{anio}/{mes:02d}"
)

r = requests.get(
    url,
    params={
        "usuario": DMC_USUARIO,
        "token": DMC_TOKEN
    },
    timeout=30
)

print("HTTP:", r.status_code)

datos_hist = r.json()

print(datos_hist.keys())
print("Registros:", datos_hist.get("registros"))
print("Status:", datos_hist.get("status"))

HTTP: 200
dict_keys(['organismo', 'pais', 'fechaCreacion', 'timezone', 'producto', 'registros', 'status', 'datosEstaciones'])
Registros: 2976
Status: Estación con datos disponibles recientemente


In [52]:
datos_mes = datos_hist["datosEstaciones"]["datos"]

df_mes = pd.DataFrame(datos_mes)

print("Filas:", len(df_mes))
print("Columnas:", df_mes.columns.tolist())

display(df_mes.head())

Filas: 2976
Columnas: ['momento', 'temperatura', 'temperatura02Mts', 'temperatura10Mts', 'temperatura30Mts', 'puntoDeRocio', 'temperaturaMinima12Horas', 'temperaturaMaxima12Horas', 'humedadRelativa', 'radiacionGlobalInst', 'presionEstacion', 'presionNivelDelMar', 'presionNivelEstandar', 'aguaCaidaDelMinuto', 'aguaCaida6Horas', 'aguaCaida24Horas', 'direccionDelViento', 'fuerzaDelViento', 'direccionDelVientoPromedio2Minutos', 'fuerzaDelVientoPromedio2Minutos', 'direccionDelVientoPromedio10Minutos', 'fuerzaDelVientoPromedio10Minutos', 'direccionDelViento02MinutosMax', 'fuerzaDelViento02MinutosMax', 'direccionDelViento10MinutosMax', 'fuerzaDelViento10MinutosMax']


,momento,temperatura,temperatura02Mts,temperatura10Mts,temperatura30Mts,puntoDeRocio,temperaturaMinima12Horas,temperaturaMaxima12Horas,humedadRelativa,radiacionGlobalInst,...,direccionDelViento,fuerzaDelViento,direccionDelVientoPromedio2Minutos,fuerzaDelVientoPromedio2Minutos,direccionDelVientoPromedio10Minutos,fuerzaDelVientoPromedio10Minutos,direccionDelViento02MinutosMax,fuerzaDelViento02MinutosMax,direccionDelViento10MinutosMax,fuerzaDelViento10MinutosMax
0,2024-01-01 00:00:00,16.9 °C,None,None,None,11.8 °C,16.7 °C,24.2 °C,71 %,0.000 Watt/m2,...,234 °,0.1 kt,148 °,0.2 kt,167 °,1.5 kt,None,None,None,None
1,2024-01-01 00:15:00,16.4 °C,None,None,None,11.7 °C,16.7 °C,24.2 °C,73 %,0.000 Watt/m2,...,170 °,0.6 kt,158 °,1.3 kt,162 °,2.0 kt,None,None,None,None
2,2024-01-01 00:30:00,15.8 °C,None,None,None,11.6 °C,16.7 °C,24.2 °C,76 %,0.000 Watt/m2,...,128 °,2.0 kt,147 °,2.5 kt,159 °,1.8 kt,None,None,None,None
3,2024-01-01 00:45:00,15.3 °C,None,None,None,11.6 °C,16.7 °C,24.2 °C,78 %,0.000 Watt/m2,...,143 °,1.5 kt,159 °,1.3 kt,159 °,2.3 kt,None,None,None,None
4,2024-01-01 01:00:00,14.8 °C,None,None,None,11.5 °C,16.7 °C,24.2 °C,80 %,0.000 Watt/m2,...,90 °,2.9 kt,144 °,2.5 kt,143 °,1.8 kt,None,None,None,None


In [53]:
df_mes["momento_dt"] = pd.to_datetime(
    df_mes["momento"],
    errors="coerce"
)

print("Primera observación:", df_mes["momento_dt"].min())
print("Última observación:", df_mes["momento_dt"].max())

Primera observación: 2024-01-01 00:00:00
Última observación: 2024-01-31 23:45:00


In [54]:
def limpiar_numero(serie):
    return pd.to_numeric(
        serie.astype(str)
        .str.extract(r"([-+]?\d*\.?\d+)")[0],
        errors="coerce"
    )

In [55]:
def obtener_mes_dmc(codigo, anio, mes):
    url = (
        "https://climatologia.meteochile.gob.cl"
        f"/application/servicios/getDatosRecientesEma/"
        f"{codigo}/{anio}/{mes:02d}"
    )

    try:
        r = requests.get(
            url,
            params={
                "usuario": DMC_USUARIO,
                "token": DMC_TOKEN
            },
            timeout=30
        )

        if r.status_code != 200:
            return pd.DataFrame()

        data = r.json()

        datos = (
            data.get("datosEstaciones", {})
            .get("datos", [])
        )

        if not datos:
            return pd.DataFrame()

        return pd.DataFrame(datos)

    except Exception:
        return pd.DataFrame()

In [56]:
def revisar_mes_dmc(df):
    necesarias = [
        "temperatura",
        "humedadRelativa",
        "fuerzaDelViento"
    ]

    if df.empty:
        return None

    if not all(c in df.columns for c in necesarias):
        return None

    temp = limpiar_numero(df["temperatura"])
    hum = limpiar_numero(df["humedadRelativa"])
    viento = limpiar_numero(df["fuerzaDelViento"])

    return {
        "registros": len(df),
        "temp_validos": temp.notna().sum(),
        "hum_validos": hum.notna().sum(),
        "viento_validos": viento.notna().sum(),
        "temp_pct": temp.notna().mean() * 100,
        "hum_pct": hum.notna().mean() * 100,
        "viento_pct": viento.notna().mean() * 100
    }

In [57]:
resultados_inicio = []

for codigo, nombre in ESTACIONES.items():

    print(f"\nBuscando: {codigo} - {nombre}")

    encontrado = False

    for anio in range(2010, 2027):
        for mes in range(1, 13):

            # No buscar meses futuros
            if anio == 2026 and mes > 10:
                break

            df = obtener_mes_dmc(codigo, anio, mes)
            info = revisar_mes_dmc(df)

            if info is None:
                continue

            # Consideramos útil si las 3 variables tienen
            # al menos 80 % de observaciones válidas
            if (
                info["temp_pct"] >= 80
                and info["hum_pct"] >= 80
                and info["viento_pct"] >= 80
            ):
                resultados_inicio.append({
                    "codigo": codigo,
                    "estacion": nombre,
                    "primer_mes_util": f"{anio}-{mes:02d}",
                    **info
                })

                print(
                    f"Primer mes útil: {anio}-{mes:02d} "
                    f"| Temp {info['temp_pct']:.1f}% "
                    f"| Hum {info['hum_pct']:.1f}% "
                    f"| Viento {info['viento_pct']:.1f}%"
                )

                encontrado = True
                break

            time.sleep(0.05)

        if encontrado:
            break

resumen_inicio_dmc = pd.DataFrame(resultados_inicio)

display(resumen_inicio_dmc)


Buscando: 320019 - San Felipe Escuela Agrícola
Primer mes útil: 2017-10 | Temp 100.0% | Hum 100.0% | Viento 100.0%

Buscando: 320041 - Viña del Mar Ad. (Torquemada)
Primer mes útil: 2018-11 | Temp 100.0% | Hum 100.0% | Viento 100.0%

Buscando: 330007 - Rodelillo, Ad.
Primer mes útil: 2014-11 | Temp 100.0% | Hum 100.0% | Viento 100.0%

Buscando: 330030 - Santo Domingo, Ad.
Primer mes útil: 2014-08 | Temp 100.0% | Hum 100.0% | Viento 100.0%


,codigo,estacion,primer_mes_util,registros,temp_validos,hum_validos,viento_validos,temp_pct,hum_pct,viento_pct
0,320019,San Felipe Escuela Agrícola,2017-10,2,2,2,2,100.0,100.0,100.0
1,320041,Viña del Mar Ad. (Torquemada),2018-11,29,29,29,29,100.0,100.0,100.0
2,330007,"Rodelillo, Ad.",2014-11,1821,1821,1821,1821,100.0,100.0,100.0
3,330030,"Santo Domingo, Ad.",2014-08,1021,1021,1021,1021,100.0,100.0,100.0


In [58]:
import calendar

def revisar_mes_dmc(df, anio, mes):
    if df.empty:
        return None

    necesarias = [
        "temperatura",
        "humedadRelativa",
        "fuerzaDelViento"
    ]

    if not all(c in df.columns for c in necesarias):
        return None

    temp = limpiar_numero(df["temperatura"])
    hum = limpiar_numero(df["humedadRelativa"])
    viento = limpiar_numero(df["fuerzaDelViento"])

    validos_3 = temp.notna() & hum.notna() & viento.notna()

    dias_mes = calendar.monthrange(anio, mes)[1]
    esperados = dias_mes * 24 * 4

    return {
        "registros": len(df),
        "esperados": esperados,
        "validos_3": int(validos_3.sum()),
        "cobertura_pct": validos_3.sum() / esperados * 100
    }

In [59]:
resultados_inicio = []

for codigo, nombre in ESTACIONES.items():

    print(f"\nBuscando: {codigo} - {nombre}")

    encontrado = False

    for anio in range(2010, 2027):
        for mes in range(1, 13):

            if anio == 2026 and mes > 10:
                break

            df = obtener_mes_dmc(codigo, anio, mes)
            info = revisar_mes_dmc(df, anio, mes)

            if info is None:
                continue

            if info["cobertura_pct"] >= 80:

                resultados_inicio.append({
                    "codigo": codigo,
                    "estacion": nombre,
                    "primer_mes_util": f"{anio}-{mes:02d}",
                    **info
                })

                print(
                    f"Primer mes útil: {anio}-{mes:02d} | "
                    f"{info['validos_3']}/{info['esperados']} | "
                    f"Cobertura {info['cobertura_pct']:.1f}%"
                )

                encontrado = True
                break

        if encontrado:
            break

resumen_inicio_dmc = pd.DataFrame(resultados_inicio)
display(resumen_inicio_dmc)


Buscando: 320019 - San Felipe Escuela Agrícola
Primer mes útil: 2019-06 | 2578/2880 | Cobertura 89.5%

Buscando: 320041 - Viña del Mar Ad. (Torquemada)
Primer mes útil: 2018-12 | 2975/2976 | Cobertura 100.0%

Buscando: 330007 - Rodelillo, Ad.
Primer mes útil: 2014-12 | 2973/2976 | Cobertura 99.9%

Buscando: 330030 - Santo Domingo, Ad.
Primer mes útil: 2014-09 | 2399/2880 | Cobertura 83.3%


,codigo,estacion,primer_mes_util,registros,esperados,validos_3,cobertura_pct
0,320019,San Felipe Escuela Agrícola,2019-06,2578,2880,2578,89.513889
1,320041,Viña del Mar Ad. (Torquemada),2018-12,2975,2976,2975,99.966398
2,330007,"Rodelillo, Ad.",2014-12,2973,2976,2973,99.899194
3,330030,"Santo Domingo, Ad.",2014-09,2399,2880,2399,83.298611


In [60]:
resultados_cobertura = []

for codigo, nombre in ESTACIONES.items():

    print(f"\nAnalizando {codigo} - {nombre}")

    for anio in range(2019, 2026):
        for mes in range(1, 13):

            # Comenzar en junio de 2019
            if anio == 2019 and mes < 6:
                continue

            # CONAF termina en mayo de 2025
            if anio == 2025 and mes > 5:
                break

            df = obtener_mes_dmc(codigo, anio, mes)
            info = revisar_mes_dmc(df, anio, mes)

            cobertura = 0 if info is None else info["cobertura_pct"]

            resultados_cobertura.append({
                "codigo": codigo,
                "estacion": nombre,
                "periodo": f"{anio}-{mes:02d}",
                "cobertura_pct": cobertura
            })

cobertura_dmc = pd.DataFrame(resultados_cobertura)

display(cobertura_dmc.head(20))


Analizando 320019 - San Felipe Escuela Agrícola

Analizando 320041 - Viña del Mar Ad. (Torquemada)

Analizando 330007 - Rodelillo, Ad.

Analizando 330030 - Santo Domingo, Ad.


,codigo,estacion,periodo,cobertura_pct
0,320019,San Felipe Escuela Agrícola,2019-06,89.513889
1,320019,San Felipe Escuela Agrícola,2019-07,100.000000
2,320019,San Felipe Escuela Agrícola,2019-08,100.000000
3,320019,San Felipe Escuela Agrícola,2019-09,100.000000
4,320019,San Felipe Escuela Agrícola,2019-10,100.000000
5,320019,San Felipe Escuela Agrícola,2019-11,100.000000
6,320019,San Felipe Escuela Agrícola,2019-12,100.000000
7,320019,San Felipe Escuela Agrícola,2020-01,100.000000
8,320019,San Felipe Escuela Agrícola,2020-02,100.000000
9,320019,San Felipe Escuela Agrícola,2020-03,100.000000


In [61]:
resumen_cobertura = (
    cobertura_dmc
    .groupby(["codigo", "estacion"])
    .agg(
        meses=("periodo", "count"),
        cobertura_media=("cobertura_pct", "mean"),
        cobertura_minima=("cobertura_pct", "min"),
        meses_80_o_mas=("cobertura_pct", lambda x: (x >= 80).sum()),
        meses_menor_80=("cobertura_pct", lambda x: (x < 80).sum()),
        meses_sin_datos=("cobertura_pct", lambda x: (x == 0).sum())
    )
    .reset_index()
)

display(resumen_cobertura)

,codigo,estacion,meses,cobertura_media,cobertura_minima,meses_80_o_mas,meses_menor_80,meses_sin_datos
0,320019,San Felipe Escuela Agrícola,72,97.772691,57.896505,69,3,0
1,320041,Viña del Mar Ad. (Torquemada),72,83.401861,0.000000,57,15,9
2,330007,"Rodelillo, Ad.",72,98.888343,84.173387,72,0,0
3,330030,"Santo Domingo, Ad.",72,97.564530,69.993280,71,1,0


In [62]:
problemas = cobertura_dmc[
    cobertura_dmc["cobertura_pct"] < 80
]

display(problemas)

,codigo,estacion,periodo,cobertura_pct
15,320019,San Felipe Escuela Agrícola,2020-09,68.541667
26,320019,San Felipe Escuela Agrícola,2021-08,57.896505
58,320019,San Felipe Escuela Agrícola,2024-04,73.750000
81,320041,Viña del Mar Ad. (Torquemada),2020-03,72.513441
82,320041,Viña del Mar Ad. (Torquemada),2020-04,57.708333
83,320041,Viña del Mar Ad. (Torquemada),2020-05,61.155914
84,320041,Viña del Mar Ad. (Torquemada),2020-06,0.000000
85,320041,Viña del Mar Ad. (Torquemada),2020-07,0.000000
86,320041,Viña del Mar Ad. (Torquemada),2020-08,0.000000
87,320041,Viña del Mar Ad. (Torquemada),2020-09,0.000000


In [63]:
resultados_diarios = []

for codigo, nombre in ESTACIONES.items():

    print(f"Analizando {codigo} - {nombre}")

    for anio in range(2019, 2026):
        for mes in range(1, 13):

            if anio == 2019 and mes < 6:
                continue

            if anio == 2025 and mes > 5:
                break

            df = obtener_mes_dmc(codigo, anio, mes)

            if df.empty:
                continue

            necesarias = [
                "momento",
                "temperatura",
                "humedadRelativa",
                "fuerzaDelViento"
            ]

            if not all(c in df.columns for c in necesarias):
                continue

            df = df.copy()

            df["momento_dt"] = pd.to_datetime(
                df["momento"],
                errors="coerce"
            )

            df["fecha"] = df["momento_dt"].dt.date

            df["temp"] = limpiar_numero(df["temperatura"])
            df["hum"] = limpiar_numero(df["humedadRelativa"])
            df["viento"] = limpiar_numero(df["fuerzaDelViento"])

            df["valido_3"] = (
                df["temp"].notna()
                & df["hum"].notna()
                & df["viento"].notna()
            )

            diario = (
                df.groupby("fecha")
                .agg(
                    registros=("momento", "size"),
                    validos_3=("valido_3", "sum")
                )
                .reset_index()
            )

            # 96 observaciones esperadas por día:
            # 24 h × 4 observaciones/h
            diario["cobertura_pct"] = (
                diario["validos_3"] / 96 * 100
            )

            diario["codigo"] = codigo
            diario["estacion"] = nombre

            resultados_diarios.append(diario)

cobertura_diaria = pd.concat(
    resultados_diarios,
    ignore_index=True
)

Analizando 320019 - San Felipe Escuela Agrícola
Analizando 320041 - Viña del Mar Ad. (Torquemada)
Analizando 330007 - Rodelillo, Ad.
Analizando 330030 - Santo Domingo, Ad.


In [64]:
resumen_diario = (
    cobertura_diaria
    .groupby(["codigo", "estacion"])
    .agg(
        dias_con_algún_dato=("fecha", "count"),
        dias_80_o_mas=(
            "cobertura_pct",
            lambda x: (x >= 80).sum()
        ),
        dias_menor_80=(
            "cobertura_pct",
            lambda x: (x < 80).sum()
        ),
        cobertura_media=("cobertura_pct", "mean")
    )
    .reset_index()
)

display(resumen_diario)

,codigo,estacion,dias_con_algún_dato,dias_80_o_mas,dias_menor_80,cobertura_media
0,320019,San Felipe Escuela Agrícola,2169,2116,53,98.811376
1,320041,Viña del Mar Ad. (Torquemada),1866,1813,53,98.004310
2,330007,"Rodelillo, Ad.",2188,2153,35,99.066404
3,330030,"Santo Domingo, Ad.",2181,2144,37,98.060427


In [65]:
torquemada = cobertura_diaria[
    cobertura_diaria["codigo"] == "320041"
]

display(
    torquemada[
        torquemada["cobertura_pct"] < 80
    ].head(100)
)

,fecha,registros,validos_3,cobertura_pct,codigo,estacion
2466,2020-03-24,50,50,52.083333,320041,Viña del Mar Ad. (Torquemada)
2467,2020-04-13,30,30,31.250000,320041,Viña del Mar Ad. (Torquemada)
2504,2021-03-23,25,25,26.041667,320041,Viña del Mar Ad. (Torquemada)
3214,2023-03-03,72,72,75.000000,320041,Viña del Mar Ad. (Torquemada)
3216,2023-03-05,75,75,78.125000,320041,Viña del Mar Ad. (Torquemada)
3217,2023-03-06,71,71,73.958333,320041,Viña del Mar Ad. (Torquemada)
3218,2023-03-07,72,72,75.000000,320041,Viña del Mar Ad. (Torquemada)
3221,2023-03-10,76,76,79.166667,320041,Viña del Mar Ad. (Torquemada)
3610,2024-04-02,96,62,64.583333,320041,Viña del Mar Ad. (Torquemada)
3611,2024-04-03,96,43,44.791667,320041,Viña del Mar Ad. (Torquemada)


In [41]:
codigo = "330007"
anio = 2024
mes = 1

url = (
    "https://climatologia.meteochile.gob.cl"
    f"/application/servicios/getDatosRecientesEma/"
    f"{codigo}/{anio}/{mes:02d}"
)

r = requests.get(
    url,
    params={
        "usuario": DMC_USUARIO,
        "token": DMC_TOKEN
    }
)

print(r.status_code)

datos = r.json()

print(
    json.dumps(
        datos,
        ensure_ascii=False,
        indent=2
    )[:5000]
)

200
{
  "organismo": "Dirección Meteorlógica de Chile",
  "pais": "Chile",
  "fechaCreacion": "05-10-2026 21:50",
  "timezone": "UTC",
  "producto": "Datos Recientes de la estación automática últimas 12 horas",
  "registros": 2976,
  "status": "Estación con datos disponibles recientemente",
  "datosEstaciones": {
    "estacion": {
      "codigoNacional": "330007",
      "codigoOMM": "85560",
      "codigoOACI": "SCRD",
      "nombreEstacion": "Rodelillo, Ad.",
      "latitud": "-33.06528",
      "longitud": "-71.55639",
      "altura": 335
    },
    "datos": [
      {
        "momento": "2024-01-01 00:00:00",
        "temperatura": "16.9 °C",
        "temperatura02Mts": null,
        "temperatura10Mts": null,
        "temperatura30Mts": null,
        "puntoDeRocio": "11.8 °C",
        "temperaturaMinima12Horas": "16.7 °C",
        "temperaturaMaxima12Horas": "24.2 °C",
        "humedadRelativa": "71 %",
        "radiacionGlobalInst": " 0.000 Watt/m2",
        "presionEstacion": "979.9

In [42]:
def buscar_registros(obj):
    encontrados = []

    if isinstance(obj, dict):
        if "momento" in obj:
            encontrados.append(obj)

        for valor in obj.values():
            encontrados.extend(buscar_registros(valor))

    elif isinstance(obj, list):
        for item in obj:
            encontrados.extend(buscar_registros(item))

    return encontrados

In [43]:
registros = buscar_registros(datos)

df_dmc = pd.DataFrame(registros)

print(df_dmc.shape)
display(df_dmc.head())
print(df_dmc.columns.tolist())

(2976, 26)


,momento,temperatura,temperatura02Mts,temperatura10Mts,temperatura30Mts,puntoDeRocio,temperaturaMinima12Horas,temperaturaMaxima12Horas,humedadRelativa,radiacionGlobalInst,...,direccionDelViento,fuerzaDelViento,direccionDelVientoPromedio2Minutos,fuerzaDelVientoPromedio2Minutos,direccionDelVientoPromedio10Minutos,fuerzaDelVientoPromedio10Minutos,direccionDelViento02MinutosMax,fuerzaDelViento02MinutosMax,direccionDelViento10MinutosMax,fuerzaDelViento10MinutosMax
0,2024-01-01 00:00:00,16.9 °C,None,None,None,11.8 °C,16.7 °C,24.2 °C,71 %,0.000 Watt/m2,...,234 °,0.1 kt,148 °,0.2 kt,167 °,1.5 kt,None,None,None,None
1,2024-01-01 00:15:00,16.4 °C,None,None,None,11.7 °C,16.7 °C,24.2 °C,73 %,0.000 Watt/m2,...,170 °,0.6 kt,158 °,1.3 kt,162 °,2.0 kt,None,None,None,None
2,2024-01-01 00:30:00,15.8 °C,None,None,None,11.6 °C,16.7 °C,24.2 °C,76 %,0.000 Watt/m2,...,128 °,2.0 kt,147 °,2.5 kt,159 °,1.8 kt,None,None,None,None
3,2024-01-01 00:45:00,15.3 °C,None,None,None,11.6 °C,16.7 °C,24.2 °C,78 %,0.000 Watt/m2,...,143 °,1.5 kt,159 °,1.3 kt,159 °,2.3 kt,None,None,None,None
4,2024-01-01 01:00:00,14.8 °C,None,None,None,11.5 °C,16.7 °C,24.2 °C,80 %,0.000 Watt/m2,...,90 °,2.9 kt,144 °,2.5 kt,143 °,1.8 kt,None,None,None,None


['momento', 'temperatura', 'temperatura02Mts', 'temperatura10Mts', 'temperatura30Mts', 'puntoDeRocio', 'temperaturaMinima12Horas', 'temperaturaMaxima12Horas', 'humedadRelativa', 'radiacionGlobalInst', 'presionEstacion', 'presionNivelDelMar', 'presionNivelEstandar', 'aguaCaidaDelMinuto', 'aguaCaida6Horas', 'aguaCaida24Horas', 'direccionDelViento', 'fuerzaDelViento', 'direccionDelVientoPromedio2Minutos', 'fuerzaDelVientoPromedio2Minutos', 'direccionDelVientoPromedio10Minutos', 'fuerzaDelVientoPromedio10Minutos', 'direccionDelViento02MinutosMax', 'fuerzaDelViento02MinutosMax', 'direccionDelViento10MinutosMax', 'fuerzaDelViento10MinutosMax']


In [44]:
columnas = [
    c for c in [
        "momento",
        "temperatura",
        "humedadRelativa",
        "fuerzaDelViento",
        "fuerzaDelVientoPromedio10Minutos",
        "fuerzaDelViento10MinutosMax"
    ]
    if c in df_dmc.columns
]

display(df_dmc[columnas].head(30))

,momento,temperatura,humedadRelativa,fuerzaDelViento,fuerzaDelVientoPromedio10Minutos,fuerzaDelViento10MinutosMax
0,2024-01-01 00:00:00,16.9 °C,71 %,0.1 kt,1.5 kt,None
1,2024-01-01 00:15:00,16.4 °C,73 %,0.6 kt,2.0 kt,None
2,2024-01-01 00:30:00,15.8 °C,76 %,2.0 kt,1.8 kt,None
3,2024-01-01 00:45:00,15.3 °C,78 %,1.5 kt,2.3 kt,None
4,2024-01-01 01:00:00,14.8 °C,80 %,2.9 kt,1.8 kt,None
5,2024-01-01 01:15:00,14.5 °C,82 %,1.6 kt,2.2 kt,None
6,2024-01-01 01:30:00,14.3 °C,83 %,1.8 kt,2.5 kt,None
7,2024-01-01 01:45:00,14.0 °C,85 %,1.7 kt,1.6 kt,None
8,2024-01-01 02:00:00,14.3 °C,84 %,2.2 kt,2.1 kt,None
9,2024-01-01 02:15:00,14.0 °C,85 %,2.4 kt,1.7 kt,None


In [45]:
def obtener_mes_dmc(codigo, anio, mes):
    url = (
        "https://climatologia.meteochile.gob.cl"
        f"/application/servicios/getDatosRecientesEma/"
        f"{codigo}/{anio}/{mes:02d}"
    )

    r = requests.get(
        url,
        params={
            "usuario": DMC_USUARIO,
            "token": DMC_TOKEN
        },
        timeout=30
    )

    if r.status_code != 200:
        return pd.DataFrame()

    try:
        data = r.json()
    except:
        return pd.DataFrame()

    registros = buscar_registros(data)

    if not registros:
        return pd.DataFrame()

    return pd.DataFrame(registros)

In [46]:
def mes_completo(df):
    necesarias = [
        "temperatura",
        "humedadRelativa",
        "fuerzaDelViento"
    ]

    if df.empty:
        return False

    if not all(c in df.columns for c in necesarias):
        return False

    return all(df[c].notna().any() for c in necesarias)

In [47]:
resumen_estaciones = []

for codigo, nombre in ESTACIONES.items():

    primera_fecha = None

    print(f"\nBuscando {codigo} - {nombre}")

    for anio in range(2010, 2027):

        encontrado = False

        for mes in range(1, 13):

            df = obtener_mes_dmc(codigo, anio, mes)

            if mes_completo(df):

                momentos = pd.to_datetime(
                    df["momento"],
                    errors="coerce"
                )

                primera_fecha = momentos.min()

                print(
                    "Primera disponibilidad completa:",
                    primera_fecha
                )

                encontrado = True
                break

            time.sleep(0.1)

        if encontrado:
            break

    resumen_estaciones.append({
        "codigo": codigo,
        "estacion": nombre,
        "inicio_temp_hum_viento": primera_fecha
    })

resumen_estaciones = pd.DataFrame(resumen_estaciones)

display(resumen_estaciones)


Buscando 320019 - San Felipe Escuela Agrícola
Primera disponibilidad completa: 2017-10-19 01:15:00

Buscando 320041 - Viña del Mar Ad. (Torquemada)
Primera disponibilidad completa: 2018-11-30 16:45:14

Buscando 330007 - Rodelillo, Ad.
Primera disponibilidad completa: 2014-11-06 17:15:00

Buscando 330030 - Santo Domingo, Ad.
Primera disponibilidad completa: 2014-08-20 15:45:01


,codigo,estacion,inicio_temp_hum_viento
0,320019,San Felipe Escuela Agrícola,2017-10-19 01:15:00
1,320041,Viña del Mar Ad. (Torquemada),2018-11-30 16:45:14
2,330007,"Rodelillo, Ad.",2014-11-06 17:15:00
3,330030,"Santo Domingo, Ad.",2014-08-20 15:45:01


# IDE

In [74]:
BASE_IDE = (
    "https://esri.ciren.cl/server/rest/services/"
    "SIMEF/CAMBIO_USO_DE_LA_TIERRA_1/MapServer"
)

meta_ide = requests.get(
    BASE_IDE,
    params={"f": "json"}
).json()

capas_ide = pd.DataFrame(meta_ide["layers"])

display(capas_ide)

,id,name,parentLayerId,defaultVisibility,subLayerIds,minScale,maxScale,type,geometryType,supportsDynamicLegends
0,0,REGIÓN ARICA Y PARINACOTA (CONAF 2012-2021),-1,False,None,0,0,Feature Layer,esriGeometryPolygon,True
1,1,REGIÓN TARAPACÁ (CONAF 2013-2021),-1,False,None,0,0,Feature Layer,esriGeometryPolygon,True
2,2,REGIÓN ATACAMA (2015-2021),-1,False,None,0,0,Feature Layer,esriGeometryPolygon,True
3,3,REGIÓN COQUIMBO (CONAF 2001-2023),-1,False,None,0,0,Feature Layer,esriGeometryPolygon,True
4,4,REGIÓN VALPARAÍSO (CONAF 2001-2023),-1,False,None,0,0,Feature Layer,esriGeometryPolygon,True
5,5,REGIÓN METROPOLITANA (CONAF 2001-2023),-1,False,None,0,0,Feature Layer,esriGeometryPolygon,True
6,6,REGIÓN LIBERTADOR GENERAL BERNARDO O'HIGGINS (CONAF 2001-2023),-1,False,None,0,0,Feature Layer,esriGeometryPolygon,True
7,7,REGIÓN DEL MAULE (CONAF 2001-2023),-1,False,None,0,0,Feature Layer,esriGeometryPolygon,True
8,8,REGIÓN DE ÑUBLE (CONAF 2001-2023),-1,False,None,0,0,Feature Layer,esriGeometryPolygon,True
9,9,REGIÓN DEL BIOBÍO (CONAF 2001-2023),-1,False,None,0,0,Feature Layer,esriGeometryPolygon,True


In [75]:
LAYER_IDE = 4

meta_valpo = requests.get(
    f"{BASE_IDE}/{LAYER_IDE}",
    params={"f": "json"}
).json()

campos_ide = pd.DataFrame(meta_valpo["fields"])

display(
    campos_ide[
        ["name", "alias", "type"]
    ]
)

,name,alias,type
0,sub_ipcc01,Descripción Sub Uso IPCC al año 2001,esriFieldTypeString
1,sub_ipcc13,Descripción Sub Uso IPCC al año 2013,esriFieldTypeString
2,sub_ipcc16,Descripción Sub Uso IPCC al año 2016,esriFieldTypeString
3,sub_ipcc17,Descripción Sub Uso IPCC al año 2017,esriFieldTypeString
4,sub_ipcc19,Descripción Sub Uso IPCC al año 2019,esriFieldTypeString
5,sub_ipcc21,Descripción Sub Uso IPCC al año 2021,esriFieldTypeString
6,sub_ipcc23,Descripción Sub Uso IPCC al año 2023,esriFieldTypeString
7,uso_ipcc01,Descripción Uso IPCC al año 2001,esriFieldTypeString
8,uso_ipcc13,Descripción Uso IPCC al año 2013,esriFieldTypeString
9,uso_ipcc16,Descripción Uso IPCC al año 2016,esriFieldTypeString


In [76]:
url = f"{BASE_IDE}/{LAYER_IDE}/query"

params = {
    "where": "1=1",
    "outFields": "*",
    "returnGeometry": "false",
    "resultRecordCount": 10,
    "f": "json"
}

respuesta = requests.get(
    url,
    params=params
).json()

df_ide = pd.DataFrame(
    [x["attributes"] for x in respuesta["features"]]
)

print(df_ide.shape)
display(df_ide)
print(df_ide.columns.tolist())

(10, 53)


,sub_ipcc01,sub_ipcc13,sub_ipcc16,sub_ipcc17,sub_ipcc19,sub_ipcc21,sub_ipcc23,uso_ipcc01,uso_ipcc13,uso_ipcc16,...,s_tf_21,s_tf_23,sup_ha,t_f_01,t_f_13,t_f_16,t_f_17,t_f_19,t_f_21,t_f_23
0,Bosque Nativo,Bosque Nativo,Bosque Nativo,Bosque Nativo,Bosque Nativo,Bosque Nativo,Tierras de Cultivo,Tierras Forestales,Tierras Forestales,Tierras Forestales,...,Espino,No Aplica,2.263263,Esclerófilo,Esclerófilo,Esclerófilo,Esclerófilo,Esclerófilo,Esclerófilo,No Aplica
1,Bosque Nativo,Bosque Nativo,Bosque Nativo,Bosque Nativo,Bosque Nativo,Bosque Nativo,Tierras de Cultivo,Tierras Forestales,Tierras Forestales,Tierras Forestales,...,Espino,No Aplica,0.981725,Esclerófilo,Esclerófilo,Esclerófilo,Esclerófilo,Esclerófilo,Esclerófilo,No Aplica
2,Bosque Nativo,Bosque Nativo,Bosque Nativo,Bosque Nativo,Bosque Nativo,Bosque Nativo,Tierras de Cultivo,Tierras Forestales,Tierras Forestales,Tierras Forestales,...,Espino,No Aplica,1.337357,Esclerófilo,Esclerófilo,Esclerófilo,Esclerófilo,Esclerófilo,Esclerófilo,No Aplica
3,Bosque Nativo,Bosque Nativo,Bosque Nativo,Bosque Nativo,Bosque Nativo,Bosque Nativo,Tierras de Cultivo,Tierras Forestales,Tierras Forestales,Tierras Forestales,...,Esclerófilo,No Aplica,1.817498,Esclerófilo,Esclerófilo,Esclerófilo,Esclerófilo,Esclerófilo,Esclerófilo,No Aplica
4,Bosque Nativo,Bosque Nativo,Bosque Nativo,Bosque Nativo,Bosque Nativo,Bosque Nativo,Tierras de Cultivo,Tierras Forestales,Tierras Forestales,Tierras Forestales,...,Esclerófilo,No Aplica,2.171689,Esclerófilo,Esclerófilo,Esclerófilo,Esclerófilo,Esclerófilo,Esclerófilo,No Aplica
5,Bosque Nativo,Bosque Nativo,Bosque Nativo,Bosque Nativo,Bosque Nativo,Bosque Nativo,Tierras de Cultivo,Tierras Forestales,Tierras Forestales,Tierras Forestales,...,Peumo-Quillay-Litre,No Aplica,1.972611,Esclerófilo,Esclerófilo,Esclerófilo,Esclerófilo,Esclerófilo,Esclerófilo,No Aplica
6,Bosque Nativo,Bosque Nativo,Bosque Nativo,Bosque Nativo,Bosque Nativo,Bosque Nativo,Tierras de Cultivo,Tierras Forestales,Tierras Forestales,Tierras Forestales,...,Peumo-Quillay-Litre,No Aplica,1.172794,Esclerófilo,Esclerófilo,Esclerófilo,Esclerófilo,Esclerófilo,Esclerófilo,No Aplica
7,Bosque Nativo,Bosque Nativo,Bosque Nativo,Bosque Nativo,Bosque Nativo,Bosque Nativo,Tierras de Cultivo,Tierras Forestales,Tierras Forestales,Tierras Forestales,...,Esclerófilo,No Aplica,1.428575,Esclerófilo,Esclerófilo,Esclerófilo,Esclerófilo,Esclerófilo,Esclerófilo,No Aplica
8,Bosque Nativo,Bosque Nativo,Bosque Nativo,Bosque Nativo,Bosque Nativo,Bosque Nativo,Tierras de Cultivo,Tierras Forestales,Tierras Forestales,Tierras Forestales,...,Esclerófilo,No Aplica,2.938408,Esclerófilo,Esclerófilo,Esclerófilo,Esclerófilo,Esclerófilo,Esclerófilo,No Aplica
9,Bosque Nativo,Bosque Nativo,Bosque Nativo,Bosque Nativo,Bosque Nativo,Bosque Nativo,Tierras de Cultivo,Tierras Forestales,Tierras Forestales,Tierras Forestales,...,Esclerófilo,No Aplica,0.418322,Esclerófilo,Esclerófilo,Esclerófilo,Esclerófilo,Esclerófilo,Esclerófilo,No Aplica


['sub_ipcc01', 'sub_ipcc13', 'sub_ipcc16', 'sub_ipcc17', 'sub_ipcc19', 'sub_ipcc21', 'sub_ipcc23', 'uso_ipcc01', 'uso_ipcc13', 'uso_ipcc16', 'uso_ipcc17', 'uso_ipcc19', 'uso_ipcc21', 'uso_ipcc23', 'd_tc_01_13', 'd_tc_13_16', 'd_tc_16_17', 'd_tc_17_19', 'd_tc_19_21', 'd_tc_21_23', 'des_uso_01', 'des_uso_13', 'des_uso_16', 'des_uso_17', 'des_uso_19', 'des_uso_21', 'des_uso_23', 'est_01', 'est_13', 'est_16', 'est_17', 'est_19', 'est_21', 'est_23', 'nom_prov', 'nom_com', 'nom_reg', 'objectid', 's_tf_01', 's_tf_13', 's_tf_16', 's_tf_17', 's_tf_19', 's_tf_21', 's_tf_23', 'sup_ha', 't_f_01', 't_f_13', 't_f_16', 't_f_17', 't_f_19', 't_f_21', 't_f_23']


In [77]:
anios = [
    "2001",
    "2013",
    "2016",
    "2017",
    "2019",
    "2021",
    "2023"
]

columnas_temporales = [
    c for c in df_ide.columns
    if any(anio in c for anio in anios)
]

columnas_temporales

[]

In [78]:
display(df_ide[columnas_temporales].head())

""
0
1
2
3
4


In [79]:
params_count = {
    "where": "1=1",
    "returnCountOnly": "true",
    "f": "json"
}

conteo = requests.get(
    f"{BASE_IDE}/{LAYER_IDE}/query",
    params=params_count
).json()

conteo

{'count': 81383}

# SHAPEFILE CONAF

In [22]:
import geopandas as gpd
import pandas as pd

ruta = "data/raw/conaf/cobertura_incendios/II_FF.shp"

gdf = gpd.read_file(ruta)

print("Total eventos:", len(gdf))

print("\nNulos en Inicio:")
print(gdf["Inicio"].isna().sum())

print("\nNulos en Comuna:")
print(gdf["Comuna"].isna().sum())

gdf["fecha_inicio"] = pd.to_datetime(
    gdf["Inicio"],
    errors="coerce",
    dayfirst=True
)


Total eventos: 10050

Nulos en Inicio:
0

Nulos en Comuna:
0


C:\Users\Tomy\AppData\Local\Temp\ipykernel_21696\2402304897.py:16: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  gdf["fecha_inicio"] = pd.to_datetime(


In [23]:
display(
    gdf[
        ["Comuna", "Inicio", "fecha_inicio", "Superficie"]
    ].head(20)
)

,Comuna,Inicio,fecha_inicio,Superficie
0,Valparaíso,4-sep-2010 17:30,2010-09-04 17:30:00,0.50
1,Puchuncavi,9-sep-2010 15:40,2010-09-09 15:40:00,0.10
2,Hijuelas,17-sep-2010 22:28,2010-09-17 22:28:00,6.00
3,Llay-Llay,17-oct-2010 13:08,2010-10-17 13:08:00,10.00
4,Viña del Mar,23-oct-2010 17:49,2010-10-23 17:49:00,1.00
5,San Antonio,26-oct-2010 16:45,2010-10-26 16:45:00,1.00
6,Valparaíso,27-oct-2010 13:30,2010-10-27 13:30:00,4.70
7,Valparaíso,31-oct-2010 15:28,2010-10-31 15:28:00,0.15
8,Casablanca,6-nov-2010 14:33,2010-11-06 14:33:00,1.15
9,Viña del Mar,6-nov-2010 16:26,2010-11-06 16:26:00,0.70


In [24]:
validos = gdf[
    gdf["fecha_inicio"].notna()
    & gdf["Comuna"].notna()
].copy()

print("Eventos utilizables:", len(validos))
print("Eventos descartados:", len(gdf) - len(validos))

Eventos utilizables: 5222
Eventos descartados: 4828


In [25]:
validos["fecha"] = validos["fecha_inicio"].dt.date

incendios_diarios = (
    validos
    .groupby(["Comuna", "fecha"])
    .size()
    .reset_index(name="cantidad_incendios")
)

display(incendios_diarios.head(20))

,Comuna,fecha,cantidad_incendios
0,Algarrobo,2011-02-13,1
1,Algarrobo,2011-03-11,1
2,Algarrobo,2011-03-12,1
3,Algarrobo,2011-09-27,1
4,Algarrobo,2011-11-06,1
5,Algarrobo,2011-11-10,1
6,Algarrobo,2011-11-24,1
7,Algarrobo,2012-02-27,1
8,Algarrobo,2012-03-02,1
9,Algarrobo,2012-11-25,1


In [26]:
gdf["fecha_inicio"] = pd.to_datetime(
    gdf["Inicio"],
    format="mixed",
    errors="coerce",
    dayfirst=True
)

print("Total eventos:", len(gdf))
print("Fechas válidas:", gdf["fecha_inicio"].notna().sum())
print("Fechas no convertibles:", gdf["fecha_inicio"].isna().sum())
print("Primera fecha:", gdf["fecha_inicio"].min())
print("Última fecha:", gdf["fecha_inicio"].max())

Total eventos: 10050
Fechas válidas: 5222
Fechas no convertibles: 4828
Primera fecha: 2010-09-04 17:30:00
Última fecha: 2025-05-30 14:38:00


In [27]:
fallidos = gdf[gdf["fecha_inicio"].isna()][
    ["Comuna", "Inicio"]
]

print("Cantidad sin convertir:", len(fallidos))
display(fallidos.head(50))

Cantidad sin convertir: 4828


,Comuna,Inicio
52,San Antonio,1-dic-2010 11:30
53,Puchuncavi,1-dic-2010 12:50
54,San Antonio,1-dic-2010 15:50
55,San Antonio,1-dic-2010 17:50
56,Concón,2-dic-2010 11:25
57,La Ligua,2-dic-2010 10:26
58,San Antonio,3-dic-2010 10:00
59,Valparaíso,3-dic-2010 12:17
60,Valparaíso,3-dic-2010 15:07
61,Puchuncavi,4-dic-2010 11:36


In [28]:
fallidos["Inicio"].value_counts().head(30)

Inicio
27-dic-2010 15:30    3
12-ene-2011 14:10    2
23-dic-2015 19:20    2
12-ene-2011 14:34    2
11-dic-2011 14:50    2
10-ene-2012 16:40    2
1-dic-2014 15:03     2
7-dic-2012 18:40     2
8-abr-2013 12:05     2
10-ene-2016 16:05    2
19-dic-2016 14:07    2
30-dic-2015 16:15    2
2-ene-2013 17:25     2
2-ene-2012 14:40     2
18-ene-2013 18:57    2
17-ene-2011 15:20    2
30-ene-2016 07:00    2
4-dic-2011 16:45     2
23-dic-2010 14:00    2
14-dic-2018 15:40    2
29-dic-2018 16:29    2
10-abr-2019 14:05    2
14-abr-2019 15:00    2
6-dic-2019 13:38     2
14-ene-2021 14:52    2
15-ene-2021 07:10    2
3-ene-2022 22:39     2
6-ene-2022 22:55     2
27-dic-2022 16:26    2
8-dic-2016 14:00     2
Name: count, dtype: int64

In [29]:
fechas = gdf["Inicio"].astype(str).str.lower().str.strip()

meses = {
    "ene": "jan",
    "abr": "apr",
    "ago": "aug",
    "dic": "dec"
}

for esp, eng in meses.items():
    fechas = fechas.str.replace(esp, eng, regex=False)

gdf["fecha_inicio"] = pd.to_datetime(
    fechas,
    format="mixed",
    dayfirst=True,
    errors="coerce"
)

print("Total:", len(gdf))
print("Convertidas:", gdf["fecha_inicio"].notna().sum())
print("Sin convertir:", gdf["fecha_inicio"].isna().sum())
print("Primera:", gdf["fecha_inicio"].min())
print("Última:", gdf["fecha_inicio"].max())

Total: 10050
Convertidas: 10050
Sin convertir: 0
Primera: 2010-09-04 17:30:00
Última: 2025-05-30 14:38:00


In [30]:
validos = gdf[
    gdf["fecha_inicio"].notna()
    & gdf["Comuna"].notna()
].copy()

print("Eventos utilizables:", len(validos))
print("Eventos descartados:", len(gdf) - len(validos))

validos["fecha"] = validos["fecha_inicio"].dt.normalize()

incendios_diarios = (
    validos
    .groupby(["Comuna", "fecha"])
    .size()
    .reset_index(name="cantidad_incendios")
)

display(incendios_diarios.head(20))

Eventos utilizables: 10050
Eventos descartados: 0


,Comuna,fecha,cantidad_incendios
0,Algarrobo,2010-12-05,1
1,Algarrobo,2011-01-03,1
2,Algarrobo,2011-01-10,1
3,Algarrobo,2011-01-16,2
4,Algarrobo,2011-01-17,1
5,Algarrobo,2011-01-31,2
6,Algarrobo,2011-02-13,1
7,Algarrobo,2011-03-11,1
8,Algarrobo,2011-03-12,1
9,Algarrobo,2011-04-02,1


In [31]:
print("Primera fecha:", incendios_diarios["fecha"].min())
print("Última fecha:", incendios_diarios["fecha"].max())
print("Días comuna con incendio:", len(incendios_diarios))

Primera fecha: 2010-09-04 00:00:00
Última fecha: 2025-05-30 00:00:00
Días comuna con incendio: 8165


# DMC OTRO

In [66]:
import requests
import pandas as pd

url = "https://climatologia.meteochile.gob.cl/application/servicios/getEstacionesRedEma"

r = requests.get(
    url,
    params={
        "usuario": DMC_USUARIO,
        "token": DMC_TOKEN
    },
    timeout=30
)

print("HTTP:", r.status_code)

datos_estaciones = r.json()

print("Cantidad estaciones:", datos_estaciones.get("estaciones"))
print(datos_estaciones.keys())

HTTP: 200
Cantidad estaciones: 149
dict_keys(['pais', 'organismo', 'fechaCreacion', 'timezone', 'producto', 'estaciones', 'datosEstacion'])


In [67]:
df_estaciones = pd.DataFrame(
    datos_estaciones["datosEstacion"]
)

display(df_estaciones)

print(df_estaciones.columns.tolist())

,codigoNacional,CodigoOMM,codigoOACI,nombreEstacion,latitud,longitud,altura,region,NombreRegion,zonaGeografica,datos
0,170001,85403,,Visviri Tenencia,-17.59500,-69.47750,4084,15,Arica y Parinacota,Cordillera,https://climatologia.meteochile.gob.cl/application/productos/datosRecientesEma/170001
1,180005,85406,SCAR,"Chacalluta, Arica Ap.",-18.35555,-70.34028,51,15,Arica y Parinacota,Litoral,https://climatologia.meteochile.gob.cl/application/productos/datosRecientesEma/180005
2,180017,85405,,Putre,-18.20000,-69.56250,3532,15,Arica y Parinacota,PreCordillera,https://climatologia.meteochile.gob.cl/application/productos/datosRecientesEma/180017
3,180018,85407,,"Defensa Civil, Arica",-18.49111,-70.30139,71,15,Arica y Parinacota,Litoral,https://climatologia.meteochile.gob.cl/application/productos/datosRecientesEma/180018
4,180042,85408,,"Cerro Sombrero, Arica",-18.51250,-70.26556,122,15,Arica y Parinacota,Litoral,https://climatologia.meteochile.gob.cl/application/productos/datosRecientesEma/180042
...,...,...,...,...,...,...,...,...,...,...,...
144,950018,,,PeÃ±Ã³n Escuela de MontaÃ±a,-79.79222,-82.98722,863,12,Magallanes y de la AntÃ¡rtica Chilena,Valle,https://climatologia.meteochile.gob.cl/application/productos/datosRecientesEma/950018
145,950019,,,Islote D' Hainaut,-63.90278,-60.79306,18,12,Magallanes y de la AntÃ¡rtica Chilena,Valle,https://climatologia.meteochile.gob.cl/application/productos/datosRecientesEma/950019
146,950020,,,Islote Alcock,-64.24000,-61.13139,102,12,Magallanes y de la AntÃ¡rtica Chilena,Valle,https://climatologia.meteochile.gob.cl/application/productos/datosRecientesEma/950020
147,950021,,,Base J,-66.00805,-65.33917,3,12,Magallanes y de la AntÃ¡rtica Chilena,Valle,https://climatologia.meteochile.gob.cl/application/productos/datosRecientesEma/950021


['codigoNacional', 'CodigoOMM', 'codigoOACI', 'nombreEstacion', 'latitud', 'longitud', 'altura', 'region', 'NombreRegion', 'zonaGeografica', 'datos']


In [68]:
df_valparaiso = df_estaciones[
    df_estaciones["NombreRegion"]
    .astype(str)
    .str.contains("Valpara", case=False, na=False)
].copy()

display(df_valparaiso)
print("Estaciones Valparaíso:", len(df_valparaiso))

,codigoNacional,CodigoOMM,codigoOACI,nombreEstacion,latitud,longitud,altura,region,NombreRegion,zonaGeografica,datos
21,270001,85469,SCIP,Mataveri Isla de Pascua Ap.,-27.15889,-109.43250,69,5,ValparaÃ­so,Litoral,https://climatologia.meteochile.gob.cl/application/productos/datosRecientesEma/270001
37,320019,85537,,San Felipe Escuela AgrÃ­cola,-32.75528,-70.70694,662,5,ValparaÃ­so,Valle,https://climatologia.meteochile.gob.cl/application/productos/datosRecientesEma/320019
38,320041,85556,SCVM,ViÃ±a del Mar Ad. (Torquemada),-32.94944,-71.47611,140,5,ValparaÃ­so,Litoral,https://climatologia.meteochile.gob.cl/application/productos/datosRecientesEma/320041
39,320045,85538,,"Llay Llay, Las PeÃ±as",-32.88139,-70.91305,423,5,ValparaÃ­so,Valle,https://climatologia.meteochile.gob.cl/application/productos/datosRecientesEma/320045
40,320049,85533,,Chincolco Liceo agrÃ­cola,-32.22250,-70.83167,665,5,ValparaÃ­so,Valle,https://climatologia.meteochile.gob.cl/application/productos/datosRecientesEma/320049
41,320051,85539,,Los Libertadores,-32.84555,-70.11917,2955,5,ValparaÃ­so,Cordillera,https://climatologia.meteochile.gob.cl/application/productos/datosRecientesEma/320051
42,320055,,,Catemu,-32.75889,-70.94278,443,5,ValparaÃ­so,Valle,https://climatologia.meteochile.gob.cl/application/productos/datosRecientesEma/320055
43,320056,85544,,"Quintero, ClimatolÃ³gica",-32.78417,-71.52278,5,5,ValparaÃ­so,Litoral,https://climatologia.meteochile.gob.cl/application/productos/datosRecientesEma/320056
44,320063,85535,,Zapallar Catapilco,-32.56139,-71.29778,81,5,ValparaÃ­so,Valle,https://climatologia.meteochile.gob.cl/application/productos/datosRecientesEma/320063
45,320121,,,El TÃ¡rtaro (Putaendo),-32.57083,-70.70417,933,5,ValparaÃ­so,Valle,https://climatologia.meteochile.gob.cl/application/productos/datosRecientesEma/320121


Estaciones Valparaíso: 19


In [69]:
display(
    df_valparaiso[
        [
            "codigoNacional",
            "nombreEstacion",
            "latitud",
            "longitud",
            "altura",
            "NombreRegion"
        ]
    ]
)

,codigoNacional,nombreEstacion,latitud,longitud,altura,NombreRegion
21,270001,Mataveri Isla de Pascua Ap.,-27.15889,-109.43250,69,ValparaÃ­so
37,320019,San Felipe Escuela AgrÃ­cola,-32.75528,-70.70694,662,ValparaÃ­so
38,320041,ViÃ±a del Mar Ad. (Torquemada),-32.94944,-71.47611,140,ValparaÃ­so
39,320045,"Llay Llay, Las PeÃ±as",-32.88139,-70.91305,423,ValparaÃ­so
40,320049,Chincolco Liceo agrÃ­cola,-32.22250,-70.83167,665,ValparaÃ­so
41,320051,Los Libertadores,-32.84555,-70.11917,2955,ValparaÃ­so
42,320055,Catemu,-32.75889,-70.94278,443,ValparaÃ­so
43,320056,"Quintero, ClimatolÃ³gica",-32.78417,-71.52278,5,ValparaÃ­so
44,320063,Zapallar Catapilco,-32.56139,-71.29778,81,ValparaÃ­so
45,320121,El TÃ¡rtaro (Putaendo),-32.57083,-70.70417,933,ValparaÃ­so


In [70]:
candidatos_torquemada = {
    "330006": "Jardín Botánico (ex Parque Fauna), Viña del Mar",
    "330007": "Rodelillo, Ad.",
    "320056": "Quintero, Climatológica",
    "320127": "Planta Transmisora Quintero, Armada"
}

In [71]:
resultados_respaldo = []

for codigo, nombre in candidatos_torquemada.items():

    print(f"\n{codigo} - {nombre}")

    for anio, mes in [
        (2020,6),(2020,7),(2020,8),(2020,9),
        (2020,10),(2020,11),(2020,12),
        (2021,1),(2021,2)
    ]:

        df = obtener_mes_dmc(codigo, anio, mes)
        info = revisar_mes_dmc(df, anio, mes)

        cobertura = 0 if info is None else info["cobertura_pct"]

        resultados_respaldo.append({
            "codigo": codigo,
            "estacion": nombre,
            "periodo": f"{anio}-{mes:02d}",
            "cobertura_pct": cobertura
        })

        print(
            f"{anio}-{mes:02d}: "
            f"{cobertura:.1f}%"
        )

respaldo_torquemada = pd.DataFrame(resultados_respaldo)

display(respaldo_torquemada)


330006 - Jardín Botánico (ex Parque Fauna), Viña del Mar
2020-06: 100.0%
2020-07: 100.0%
2020-08: 100.0%
2020-09: 100.0%
2020-10: 100.0%
2020-11: 100.0%
2020-12: 100.0%
2021-01: 100.0%
2021-02: 100.0%

330007 - Rodelillo, Ad.
2020-06: 100.0%
2020-07: 100.0%
2020-08: 99.9%
2020-09: 100.0%
2020-10: 99.6%
2020-11: 100.0%
2020-12: 100.0%
2021-01: 100.0%
2021-02: 100.0%

320056 - Quintero, Climatológica
2020-06: 100.0%
2020-07: 100.0%
2020-08: 100.0%
2020-09: 100.0%
2020-10: 100.0%
2020-11: 100.0%
2020-12: 100.0%
2021-01: 100.0%
2021-02: 100.0%

320127 - Planta Transmisora Quintero, Armada
2020-06: 0.0%
2020-07: 0.0%
2020-08: 0.0%
2020-09: 0.0%
2020-10: 0.0%
2020-11: 0.0%
2020-12: 0.0%
2021-01: 0.0%
2021-02: 0.0%


,codigo,estacion,periodo,cobertura_pct
0,330006,"Jardín Botánico (ex Parque Fauna), Viña del Mar",2020-06,100.000000
1,330006,"Jardín Botánico (ex Parque Fauna), Viña del Mar",2020-07,100.000000
2,330006,"Jardín Botánico (ex Parque Fauna), Viña del Mar",2020-08,100.000000
3,330006,"Jardín Botánico (ex Parque Fauna), Viña del Mar",2020-09,100.000000
4,330006,"Jardín Botánico (ex Parque Fauna), Viña del Mar",2020-10,100.000000
5,330006,"Jardín Botánico (ex Parque Fauna), Viña del Mar",2020-11,100.000000
6,330006,"Jardín Botánico (ex Parque Fauna), Viña del Mar",2020-12,100.000000
7,330006,"Jardín Botánico (ex Parque Fauna), Viña del Mar",2021-01,99.966398
8,330006,"Jardín Botánico (ex Parque Fauna), Viña del Mar",2021-02,100.000000
9,330007,"Rodelillo, Ad.",2020-06,100.000000


In [72]:
display(
    respaldo_torquemada
    .groupby(["codigo", "estacion"])
    ["cobertura_pct"]
    .mean()
    .sort_values(ascending=False)
)

codigo  estacion                                       
320056  Quintero, Climatológica                            100.000000
330006  Jardín Botánico (ex Parque Fauna), Viña del Mar     99.996266
330007  Rodelillo, Ad.                                      99.947730
320127  Planta Transmisora Quintero, Armada                  0.000000
Name: cobertura_pct, dtype: float64